#carga 

In [3]:
import polars as pl
df = pl.read_parquet('/home/montes/proyectos/paralela/polar_vrs_pandas/taxi_filtrado.parquet')

# Filtrado de registros

In [4]:
# Eliminar viajes con distancia, monto o pasajeros inválidos
df_filtrado = df.filter(
    (pl.col('trip_distance') > 0) & 
    (pl.col('total_amount') > 0) & 
    (pl.col('passenger_count') > 0) &
    (pl.col('passenger_count') <= 6)
)
print(f'Dataset filtrado: {df_filtrado.shape[0]:,} filas')

Dataset filtrado: 10,839,192 filas


# Manejo de datos faltantes

In [5]:
# Llenar nulos en passenger_count con la mediana
mediana_pasajeros = df_filtrado['passenger_count'].median()
df_filtrado = df_filtrado.with_columns(
    pl.col('passenger_count').fill_null(mediana_pasajeros)
)

# Eliminar filas con nulos en columnas críticas
df_filtrado = df_filtrado.drop_nulls(subset=['tpep_pickup_datetime', 'trip_distance', 'total_amount'])
print(f'Valores nulos restantes: {df_filtrado.null_count().row(0)}')

Valores nulos restantes: (0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)


#  Creación de nuevas características

In [6]:
df_features = df.with_columns([
    pl.col('tpep_pickup_datetime').str.to_datetime().dt.hour().alias('pickup_hour'),
    pl.col('tpep_pickup_datetime').str.to_datetime().dt.weekday().alias('pickup_weekday'),
    (pl.col('tpep_dropoff_datetime').str.to_datetime() - pl.col('tpep_pickup_datetime').str.to_datetime()).dt.total_seconds().alias('trip_duration_sec'),
    (pl.col('total_amount') / pl.col('trip_distance')).alias('cost_per_mile')
])

# Agregación mediante group_by

In [7]:
df_agg = df_features.group_by('pickup_hour').agg([
    pl.col('total_amount').mean().alias('avg_fare_hour'),
    pl.col('trip_distance').mean().alias('avg_dist_hour'),
    pl.col('total_amount').count().alias('trip_count_hour')
]).sort('pickup_hour')

# join 

In [8]:
df_final = df_features.join(df_agg, on='pickup_hour', how='left')

print(df_final.shape)
df_final.head()

(10906858, 26)


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,pickup_hour,pickup_weekday,trip_duration_sec,cost_per_mile,avg_fare_hour,avg_dist_hour,trip_count_hour
i64,str,str,i64,f64,f64,f64,i64,str,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i64,f64,f64,f64,u32
2,"""2016-01-01 00:00:00""","""2016-01-01 00:00:00""",2,1.1,-73.990372,40.734695,1,"""N""",-73.981842,40.732407,2,7.5,0.5,0.5,0.0,0.0,0.3,8.8,0,5,0,8.0,16.402156,3.32865,396034
2,"""2016-01-01 00:00:00""","""2016-01-01 00:00:00""",5,4.9,-73.980782,40.729912,1,"""N""",-73.944473,40.716679,1,18.0,0.5,0.5,0.0,0.0,0.3,19.3,0,5,0,3.938776,16.402156,3.32865,396034
2,"""2016-01-01 00:00:00""","""2016-01-01 00:00:00""",1,10.54,-73.98455,40.679565,1,"""N""",-73.950272,40.788925,1,33.0,0.5,0.5,0.0,0.0,0.3,34.3,0,5,0,3.254269,16.402156,3.32865,396034
2,"""2016-01-01 00:00:00""","""2016-01-01 00:00:00""",1,4.75,-73.993469,40.71899,1,"""N""",-73.962242,40.657333,2,16.5,0.0,0.5,0.0,0.0,0.3,17.3,0,5,0,3.642105,16.402156,3.32865,396034
2,"""2016-01-01 00:00:00""","""2016-01-01 00:00:00""",3,1.76,-73.960625,40.78133,1,"""N""",-73.977264,40.758514,2,8.0,0.0,0.5,0.0,0.0,0.3,8.8,0,5,0,5.0,16.402156,3.32865,396034
